# Welcome to the Day 2 Lab!


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Just before we get started --</h2>
            <span style="color:#f71;">I thought I'd take a second to point you at this page of useful resources for the course. This includes links to all the slides.<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            Please keep this bookmarked, and I'll continue to add more useful links there over time.
            </span>
        </td>
    </tr>
</table>

## First - let's talk about the Chat Completions API

1. The simplest way to call an LLM
2. It's called Chat Completions because it's saying: "here is a conversation, please predict what should come next"
3. The Chat Completions API was invented by OpenAI, but it's so popular that everybody uses it!

### We will start by calling OpenAI again - but don't worry non-OpenAI people, your time is coming!


In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

#api_key = os.getenv('DEEPSEEK_API_KEY')
api_key = os.getenv('GOOGLE_API_KEY')


API key found and looks good so far!


## Do you know what an Endpoint is?

If not, please review the Technical Foundations guide in the guides folder

And, here is an endpoint that might interest you...

In [2]:
import requests
import os

API_KEY = os.getenv("GOOGLE_API_KEY")

# Endpoint para listar modelos
url = f"https://generativelanguage.googleapis.com/v1/models?key={API_KEY}"

try:
    response = requests.get(url)
    response.raise_for_status()
    models = response.json()
    
    print("--- Modelos Disponíveis para sua Chave ---")
    for model in models['models']:
        # Filtra apenas os que aceitam gerar conteúdo
        if "generateContent" in model['supportedGenerationMethods']:
            print(f"ID: {model['name']} | Versão: {model['version']}")
            
except Exception as e:
    print(f"Erro ao listar modelos: {e}")

--- Modelos Disponíveis para sua Chave ---
ID: models/gemini-2.5-flash | Versão: 001
ID: models/gemini-2.5-pro | Versão: 2.5
ID: models/gemini-2.0-flash | Versão: 2.0
ID: models/gemini-2.0-flash-001 | Versão: 2.0
ID: models/gemini-2.0-flash-lite-001 | Versão: 2.0
ID: models/gemini-2.0-flash-lite | Versão: 2.0
ID: models/gemini-2.5-flash-lite | Versão: 001


In [3]:
import requests
import os

# 1. Pegue sua chave
API_KEY = os.getenv("GOOGLE_API_KEY")

# 2. Use o ID EXATO que apareceu na sua lista
MODEL_ID = "gemini-2.5-flash"

# 3. Endpoint v1 (Estável)
url = f"https://generativelanguage.googleapis.com/v1/models/{MODEL_ID}:generateContent?key={API_KEY}"

payload = {
    "contents": [
        {
            "parts": [
                {"text": "Explique o que é uma API em uma frase curta."}
            ]
        }
    ]
}

headers = {"Content-Type": "application/json"}

try:
    response = requests.post(url, headers=headers, json=payload)
    response.raise_for_status()
    
    data = response.json()
    # O Gemini 2.5 mantém a mesma estrutura de resposta
    print("Resposta da IA:", data["candidates"][0]["content"]["parts"][0]["text"])

except requests.exceptions.HTTPError as err:
    print(f"Erro {response.status_code}: {response.text}")
except Exception as e:
    print(f"Erro inesperado: {e}")

Resposta da IA: Uma API (Interface de Programação de Aplicações) é uma **interface que define como diferentes softwares podem se comunicar e interagir entre si.**


In [36]:
"""
# OpenAI
Endpoints:
Chat Completions Endpoint: https://api.openai.com/v1/chat/completions

Models:
gpt-5-nano (Recommended for general tasks/coding, uses V3)
----

# Deepseek
Endpoints:
Chat Completions Endpoint: https://api.deepseek.com/chat/completions
Alternative Base URL (OpenAI Compatibility): https://api.deepseek.com/v1 
 
Models:
deepseek-chat (Recommended for general tasks/coding, uses V3)
deepseek-reasoner (Recommended for complex reasoning/thinking, uses V3)

Google Gemini 1.5 Flash
Endpoints:
MODEL_ID = "gemini-1.5-flash"
url = f"https://generativelanguage.googleapis.com{MODEL_ID}:generateContent?key={api_key}"

"""
import requests
import json

headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

payload = {
    #"model": "gpt-5-nano",
    "model": "deepseek-chat",
    "messages": [
        {"role": "user", "content": "Tell me a fun fact"}]
}

MODEL_ID = "gemini-2.5-flash"
url = f"https://generativelanguage.googleapis.com/v1/models/{MODEL_ID}:generateContent?key={API_KEY}"

# Payload structure required by the Google API
payload = {
    "contents": [{
        "parts": [{"text": "Tell me a fun fact."}]
    }]
}

headers = {
    "Content-Type": "application/json"
}

payload

{'contents': [{'parts': [{'text': 'Tell me a fun fact.'}]}]}

In [37]:
response_1 = requests.post(
    #"https://api.openai.com/v1/chat/completions",
    "https://api.deepseek.com/v1/chat/completions",
    headers=headers,
    json=payload
)

response = requests.post(url, headers=headers, data=json.dumps(payload))

if response.status_code == 200:
    data = response.json()
    #  The response comes in a nested structure: candidates -> content -> parts -> text
    #print(data['candidates'][0]['content']['parts'][0]['text'])
else:
    print(f"Error {response.status_code}: {response.text}")

response.json()

{'candidates': [{'content': {'parts': [{'text': "Here's one for you:\n\nAn **octopus has three hearts**! Two pump blood through its gills, and the third circulates blood to the rest of its body. Pretty cool, right?"}],
    'role': 'model'},
   'finishReason': 'STOP',
   'index': 0}],
 'usageMetadata': {'promptTokenCount': 6,
  'candidatesTokenCount': 41,
  'totalTokenCount': 381,
  'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 6}],
  'thoughtsTokenCount': 334},
 'modelVersion': 'gemini-2.5-flash',
 'responseId': '-kfEaeyaMtboz7IPgOGNiQ8'}

In [39]:
# response.json()["choices"][0]["message"]["content"]

# For Goolge Gemini 2.5 Flash, the response structure is different
response.json()["candidates"][0]["content"]["parts"][0]["text"]

"Here's one for you:\n\nAn **octopus has three hearts**! Two pump blood through its gills, and the third circulates blood to the rest of its body. Pretty cool, right?"

# What is the openai package?

It's known as a Python Client Library.

It's nothing more than a wrapper around making this exact call to the http endpoint.

It just allows you to work with nice Python code instead of messing around with janky json objects.

But that's it. It's open-source and lightweight. Some people think it contains OpenAI model code - it doesn't!


In [ ]:
# Create OpenAI client

from openai import OpenAI
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5-nano", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

# For Google Gemini you can switch to Google’s SDK or call their API directly. See on the next cell below with the sdk

In [42]:
#!pip install google-generativeai

import google.generativeai as genai
import os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

model = genai.GenerativeModel("gemini-2.5-flash")

response = model.generate_content("Tell me a fun fact")

print(response.text)

Here's one!

Did you know that **botanically speaking, a banana is considered a berry, while a strawberry is not?**

*   **Bananas** fit the botanical definition of a berry: they develop from a single flower with one ovary and usually have several seeds (though commercial bananas have tiny, sterile seeds).
*   **Strawberries**, on the other hand, are "aggregate fruits," meaning they develop from a single flower with multiple ovaries, and the "seeds" on the outside are actually individual fruits called achenes.


## And then this great thing happened:

OpenAI's Chat Completions API was so popular, that the other model providers created endpoints that are identical.

They are known as the "OpenAI Compatible Endpoints".

For example, google made one here: https://generativelanguage.googleapis.com/v1beta/openai/

And OpenAI decided to be kind: they said, hey, you can just use the same client library that we made for GPT. We'll allow you to specify a different endpoint URL and a different key, to use another provider.

So you can use:

```python
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="AIz....")
gemini.chat.completions.create(...)
```

And to be clear - even though OpenAI is in the code, we're only using this lightweight python client library to call the endpoint - there's no OpenAI model involved here.

If you're confused, please review Guide 9 in the Guides folder!

And now let's try it!

## THIS IS OPTIONAL - but if you wish to try out Google Gemini, please visit:

https://aistudio.google.com/

And set up your API key at

https://aistudio.google.com/api-keys

And then add your key to the `.env` file, being sure to Save the .env file after you change it:

`GOOGLE_API_KEY=AIz...`


In [49]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not google_api_key.startswith("AIz"):
    print("An API key was found, but it doesn't start AIz")
else:
    print("API key found and looks good so far!")



API key found and looks good so far!


In [ ]:
# Create OpenAI client

from openai import OpenAI

gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

response = gemini.chat.completions.create(model="gemini-2.5-flash-lite", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

"Here's a fun one:\n\n**Honey never spoils.** Archaeologists have found pots of honey in ancient Egyptian tombs that are over 3,000 years old and still perfectly edible!"

In [56]:
# The response comes in a nested structure pattern instead of candiates -> content -> parts -> text
response

ChatCompletion(id='iU7Eaa7hAvaFz7IPyP7BwAE', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Here's a fun one:\n\n**Honey never spoils.** Archaeologists have found pots of honey in ancient Egyptian tombs that are over 3,000 years old and still perfectly edible!", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1774472841, model='gemini-2.5-flash-lite', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=40, prompt_tokens=6, total_tokens=46, completion_tokens_details=None, prompt_tokens_details=None))

## And Ollama also gives an OpenAI compatible endpoint

...and it's on your local machine!

If the next cell doesn't print "Ollama is running" then please open a terminal and run `ollama serve`

In [57]:
requests.get("http://localhost:11434").content

b'Ollama is running'

### Download llama3.2 from meta

Change this to llama3.2:1b if your computer is smaller.

Don't use llama3.3 or llama4! They are too big for your computer..

In [ ]:
!ollama pull llama3.2

In [58]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [59]:
# Get a fun fact

response = ollama.chat.completions.create(model="llama3.2", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

'Here\'s one:\n\nDid you know that there is a species of jellyfish that is immortal? The Turritopsis dohrnii, also known as the "immortal jellyfish," can transform its body into a younger state through a process called transdifferentiation. This means it can essentially revert back to its polyp stage and grow back into an adult again, making it theoretically "immortal."\n\nIsn\'t that just mind-blowingly cool?'

In [ ]:
# Now let's try deepseek-r1:1.5b - this is DeepSeek "distilled" into Qwen from Alibaba Cloud

!ollama pull deepseek-r1:1.5b

In [ ]:
response = ollama.chat.completions.create(model="deepseek-r1:1.5b", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

# HOMEWORK EXERCISE ASSIGNMENT

Upgrade the day 1 project to summarize a webpage to use an Open Source model running locally via Ollama rather than OpenAI

You'll be able to use this technique for all subsequent projects if you'd prefer not to use paid APIs.

**Benefits:**
1. No API charges - open-source
2. Data doesn't leave your box

**Disadvantages:**
1. Significantly less power than Frontier Model

## Recap on installation of Ollama

Simply visit [ollama.com](https://ollama.com) and install!

Once complete, the ollama server should already be running locally.  
If you visit:  
[http://localhost:11434/](http://localhost:11434/)

You should see the message `Ollama is running`.  

If not, bring up a new Terminal (Mac) or Powershell (Windows) and enter `ollama serve`  
And in another Terminal (Mac) or Powershell (Windows), enter `ollama pull llama3.2`  
Then try [http://localhost:11434/](http://localhost:11434/) again.

If Ollama is slow on your machine, try using `llama3.2:1b` as an alternative. Run `ollama pull llama3.2:1b` from a Terminal or Powershell, and change the code from `MODEL = "llama3.2"` to `MODEL = "llama3.2:1b"`

In [ ]:
# Already done with ollama at llm_engineering/week1/Student_Exercises/Day 1.ipynb